In [6]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from skimage.feature import graycomatrix, graycoprops


#! Sesuaikan dengan lokasi folder dataset dan file hasil
#! Struktur: dataset/nama_label/gambar.jpg
INPUT_FOLDER = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\Gambar Final") # Gambar
OUTPUT_CSV = Path(r"D:\00 Kuliah\TA\Experiment\Model\5 ML\hasil_fitur_konvensional.csv")

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png"
}

METADATA_COLUMNS = {
    "file_name",
    "label",
    "image_width",
    "image_height",
    # .. seterusnya merupakan hasil dari ekstraksi
}

In [7]:
def get_label(image_path: Path) -> str:
    #** Mengambil label dari subfolder pertama di bawah folder dataset
    relative_path = image_path.relative_to(INPUT_FOLDER)

    if len(relative_path.parts) > 1:
        return relative_path.parts[0]

    # Gambar di folder utama belum memiliki label subfolder.
    return "unknown"


def shannon_entropy(channel: np.ndarray) -> float:
    #** Menghitung entropi Shannon kanal dalam bit
    values = np.asarray(channel).ravel()
    _, counts = np.unique(values, return_counts=True)

    probabilities = counts.astype(np.float64) / counts.sum()
    return float(-np.sum(probabilities * np.log2(probabilities)))


def channel_peak(channel: np.ndarray) -> float:
    #** Mengambil modus, yaitu nilai kanal yang paling sering muncul.
    values, counts = np.unique(np.asarray(channel).ravel(), return_counts=True)

    # Jika frekuensi tertinggi seri, ambil nilai terkecil.
    return float(values[np.argmax(counts)])


def mean_ratio(numerator: np.ndarray, denominator: np.ndarray) -> float:
    #** Menghitung rerata rasio piksel dengan mengabaikan penyebut nol.
    valid = denominator != 0

    if not np.any(valid):
        return 0.0

    return float(np.mean(numerator[valid] / denominator[valid]))

In [8]:
def add_glcm_features(features: dict, rgb: np.ndarray) -> None:
    #** Menghitung fitur GLCM dari citra grayscale 16 tingkat.
    #** 10 fitur: 5 properti GLCM × Mean dan Std dari 4 arah.
    
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    
    #** Kuantisasi grayscale dari 256 menjadi 16 tingkat (0–15).
    gray_quantized = (gray // 16).astype(np.uint8)

    angles = [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4]

    glcm = graycomatrix(
        gray_quantized,
        distances=[1],
        angles=angles,
        levels=16,
        symmetric=True,
        normed=True,
    )

    properties = (
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
    )

    for property_name in properties:
        values_by_angle = graycoprops(glcm, property_name)[0]

        features[f"GLCM_{property_name}_mean"] = float(
            np.mean(values_by_angle)
        )
        features[f"GLCM_{property_name}_std"] = float(
            np.std(values_by_angle, ddof=0)
        )

In [9]:
def extract_features(image_path: Path) -> dict:
    #** Mengekstrak fitur dari seluruh piksel satu gambar
    with Image.open(image_path) as image:
        rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)

    height, width, _ = rgb.shape
    pixel_count = height * width

    red = rgb[:, :, 0].astype(np.float64)
    green = rgb[:, :, 1].astype(np.float64)
    blue = rgb[:, :, 2].astype(np.float64)

    channels = {
        "R": red,
        "G": green,
        "B": blue,
    }

    features = {
        "file_name": image_path.name,
        "label": get_label(image_path),
        "image_width": width,
        "image_height": height,
    }

    #? 1. RGB: Mean dan Sigma R, G, B.
    #**    6 fitur: 3 Mean + 3 Sigma.
    for channel_name, channel in channels.items():
        features[f"mean_{channel_name}"] = float(np.mean(channel))
        features[f"std_{channel_name}"] = float(
            np.std(channel, ddof=0)
        )

    # Blue Channel Value identik dengan mean_B, jadi tidak dibuat sebagai
    # kolom terpisah untuk menghindari fitur duplikat.

    #? 2. Luminansi Rec. 709.
    #**    2 fitur: Mean luminansi dan Std luminansi.
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    features["mean_luminance"] = float(np.mean(luminance))
    features["std_luminance"] = float(np.std(luminance, ddof=0))

    #? 3. HSV: Mean dan Sigma kanal H, S, V.
    #**    6 fitur.
    rgb_float = rgb.astype(np.float32) / 255.0
    hsv = cv2.cvtColor(rgb_float, cv2.COLOR_RGB2HSV)

    for index, channel_name in enumerate(("H", "S", "V")):
        channel = hsv[:, :, index]
        features[f"HSV_mean_{channel_name}"] = float(np.mean(channel))
        features[f"HSV_std_{channel_name}"] = float(
            np.std(channel, ddof=0)
        )

    #? 4. Lab: Mean dan Sigma kanal L, a, b.
    #**    6 fitur.
    lab = cv2.cvtColor(rgb_float, cv2.COLOR_RGB2LAB)

    for index, channel_name in enumerate(("L", "a", "b")):
        channel = lab[:, :, index]
        features[f"Lab_mean_{channel_name}"] = float(np.mean(channel))
        features[f"Lab_std_{channel_name}"] = float(
            np.std(channel, ddof=0)
        )

    #? 5. Peak/modus untuk 9 kanal RGB, HSV, dan Lab.
    #**    9 fitur.
    peak_channels = {
        "R": red,
        "G": green,
        "B": blue,
        "H": hsv[:, :, 0],
        "S": hsv[:, :, 1],
        "V": hsv[:, :, 2],
        "L": lab[:, :, 0],
        "a": lab[:, :, 1],
        "b": lab[:, :, 2],
    }

    for channel_name, channel in peak_channels.items():
        features[f"peak_{channel_name}"] = channel_peak(channel)

    #? 6. Rasio kanal per piksel, kemudian dirata-ratakan.
    #**    3 fitur: R/G, R/B, G/B.
    features["ratio_R_over_G"] = mean_ratio(red, green)
    features["ratio_R_over_B"] = mean_ratio(red, blue)
    features["ratio_G_over_B"] = mean_ratio(green, blue)

    #? 7. Penjumlahan dan pengurangan kanal per piksel.
    #**    6 fitur: 3 penjumlahan + 3 pengurangan.
    features["sum_R_plus_G"] = float(np.mean(red + green))
    features["sum_R_plus_B"] = float(np.mean(red + blue))
    features["sum_G_plus_B"] = float(np.mean(green + blue))

    features["diff_R_minus_G"] = float(np.mean(red - green))
    features["diff_R_minus_B"] = float(np.mean(red - blue))
    features["diff_G_minus_B"] = float(np.mean(green - blue))

    #? 8. Perkalian kombinasi kanal.
    #**    2 fitur: R×G dan R×B.
    features["product_RxG"] = float(np.mean(red * green))
    features["product_RxB"] = float(np.mean(red * blue))

    #? 9. Normalize Color: proporsi RGB per piksel.
    #**    3 fitur: Mean proporsi R, G, B.
    rgb_sum = red + green + blue
    safe_sum = np.where(rgb_sum == 0, 1.0, rgb_sum)

    normalized_channels = {
        "R": red / safe_sum,
        "G": green / safe_sum,
        "B": blue / safe_sum,
    }

    for channel_name, channel in normalized_channels.items():
        features[f"normalized_{channel_name}_mean"] = float(
            np.mean(channel)
        )

    #? 10. Log-Transformed: log(1 + intensitas) tiap kanal RGB.
    #**     3 fitur: Mean log R, G, dan B.
    features["log_R_mean"] = float(np.mean(np.log1p(red)))
    features["log_G_mean"] = float(np.mean(np.log1p(green)))
    features["log_B_mean"] = float(np.mean(np.log1p(blue)))

    #? 11. Entropi Shannon kanal RGB.
    #**     3 fitur.
    entropy = {}

    for channel_name, channel in channels.items():
        entropy[channel_name] = shannon_entropy(channel)
        features[f"entropy_{channel_name}"] = entropy[channel_name]

    #? 12. Interaksi antar-entropi.
    #**     3 fitur: R×G, R×B, dan G×B.
    features["entropy_RxG"] = entropy["R"] * entropy["G"]
    features["entropy_RxB"] = entropy["R"] * entropy["B"]
    features["entropy_GxB"] = entropy["G"] * entropy["B"]

    #? 13. Kuadrat entropi.
    #**     3 fitur: H_R², H_G², H_B².
    features["entropy_R_squared"] = entropy["R"] ** 2
    features["entropy_G_squared"] = entropy["G"] ** 2
    features["entropy_B_squared"] = entropy["B"] ** 2

    #? 14. DFT 2D: Mean power spectrum tiap kanal RGB.
    #**     3 fitur.
    for channel_name, channel in channels.items():
        spectrum = np.fft.fft2(channel)
        power_spectrum = np.abs(spectrum) ** 2
        features[f"{channel_name}_FT"] = float(np.mean(power_spectrum))

    #? 15. Interval kepercayaan 95% untuk Mean R, G, B.
    #**     6 fitur: batas bawah dan atas masing-masing kanal.
    for channel_name, channel in channels.items():
        channel_mean = float(np.mean(channel))
        sigma = float(np.std(channel, ddof=0))
        margin = 1.96 * sigma / np.sqrt(pixel_count)

        features[f"{channel_name}_CI_lower"] = channel_mean - margin
        features[f"{channel_name}_CI_upper"] = channel_mean + margin

    #? 16. GLCM: lima properti, Mean dan Std dari empat arah.
    #**     10 fitur.
    add_glcm_features(features, rgb)

    return {
        key if key in METADATA_COLUMNS else f"fitur_{key}": value
        for key, value in features.items()
    }

In [10]:
def find_images(input_folder: Path) -> list[Path]:
    #** Mencari gambar yang didukung secara rekursif di dalam folder.
    return sorted(
        path
        for path in input_folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    
if not INPUT_FOLDER.exists():
    raise FileNotFoundError(
        f"Folder input tidak ditemukan: {INPUT_FOLDER.resolve()}"
    )

image_paths = find_images(INPUT_FOLDER)

if not image_paths:
    raise FileNotFoundError(
        f"Tidak ada gambar yang didukung di {INPUT_FOLDER.resolve()}"
    )

rows = []
failed_images = []

for image_path in image_paths:
    try:
        rows.append(extract_features(image_path))
    except (OSError, UnidentifiedImageError, ValueError) as error:
        failed_images.append((image_path, str(error)))

if not rows:
    raise RuntimeError("Tidak ada gambar yang berhasil diproses.")

dataframe = pd.DataFrame(rows)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
dataframe.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

feature_columns = [
    column
    for column in dataframe.columns
    if column not in METADATA_COLUMNS
]

print(f"Gambar berhasil diproses: {len(rows)}")
print(f"Jumlah fitur: {len(feature_columns)}")
print(f"CSV tersimpan di: {OUTPUT_CSV.resolve()}")

if failed_images:
    print(f"\nGambar gagal diproses: {len(failed_images)}")
    for path, error in failed_images:
        print(f"- {path}: {error}")

Gambar berhasil diproses: 253
Jumlah fitur: 74
CSV tersimpan di: D:\00 Kuliah\TA\Experiment\Model\5 ML\hasil_fitur_konvensional.csv
